# 🚶‍♂️🚶‍♀️ YOLO + ByteTrack 다중 객체 추적
## CCTV에서 사람이 몇 명 지나갔을까? — Google Colab 실습

이 실습에서는 CCTV/보행자 영상을 입력하여 다음 기능을 구현합니다.

### 🎯 학습 목표
1. YOLO 객체 탐지 이해
2. ByteTrack 다중 객체 추적 이해
3. 프레임 사이에서 Track ID 유지
4. 사람 이동 경로(Trajectory) 시각화
5. 가상선(Line Crossing)을 이용한 사람 수 계산
6. 입장/퇴장 방향별 카운트
7. 결과 영상을 MP4로 저장
8. 통과 이벤트를 CSV로 저장

### 🧭 전체 처리 흐름

`CCTV 영상 → YOLO 사람 탐지 → ByteTrack ID 추적 → 이동 분석 → 가상선 통과 판단 → 인원 집계`

**Detection**은 현재 프레임에서 사람이 어디 있는지 찾고,  
**Tracking**은 여러 프레임에 걸쳐 같은 사람에게 동일한 ID를 유지합니다.

## 1. GPU 확인

Colab에서 **런타임 → 런타임 유형 변경 → T4 GPU**를 권장합니다.
CPU에서도 실행되지만 영상 처리가 느릴 수 있습니다.

In [ ]:
!nvidia-smi || true

import torch
print("PyTorch:", torch.__version__)
print("CUDA 사용 가능:", torch.cuda.is_available())

DEVICE = 0 if torch.cuda.is_available() else "cpu"
print("사용 장치:", DEVICE)

## 2. 라이브러리 설치

- `ultralytics` : YOLO 탐지 및 Track 모드
- `opencv-python` : 영상 입출력 및 그리기
- `pandas` : 통과 이벤트 로그 저장

In [ ]:
!pip -q install -U ultralytics opencv-python pandas

import cv2
import os
import time
import subprocess
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from collections import defaultdict
from ultralytics import YOLO

import ultralytics
print("OpenCV:", cv2.__version__)
print("Ultralytics:", ultralytics.__version__)

## 3. YOLO 모델 준비

최신 Ultralytics 경량 모델을 먼저 시도하고,
환경에서 사용할 수 없으면 자동으로 `yolo11n.pt`로 대체합니다.

실습에서는 COCO의 **person 클래스(class 0)** 만 탐지합니다.

In [ ]:
def load_yolo_model():
    candidates = ["yolo26n.pt", "yolo11n.pt"]
    last_error = None

    for model_name in candidates:
        try:
            print("모델 로딩 시도:", model_name)
            model = YOLO(model_name)
            print("✅ 모델 로딩 성공:", model_name)
            return model, model_name
        except Exception as e:
            print("로딩 실패:", model_name)
            last_error = e

    raise RuntimeError(f"YOLO 모델을 불러오지 못했습니다: {last_error}")

model, MODEL_NAME = load_yolo_model()

## 4. CCTV 동영상 업로드

사람이 화면을 지나가는 짧은 MP4 영상을 권장합니다.

- 10~60초 정도
- 사람이 2명 이상 등장
- 카메라가 고정되어 있으면 좋음
- 사람이 가상선을 위↔아래로 통과하는 장면

In [ ]:
from google.colab import files

uploaded = files.upload()

video_files = [
    name for name in uploaded.keys()
    if name.lower().endswith((".mp4", ".avi", ".mov", ".mkv", ".webm"))
]

if not video_files:
    raise ValueError("MP4/AVI/MOV/MKV/WebM 동영상을 업로드해 주세요.")

VIDEO_PATH = video_files[0]
print("사용 동영상:", VIDEO_PATH)

## 5. 영상 정보 확인

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)

fps = cap.get(cv2.CAP_PROP_FPS)
frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))

duration = frame_count / fps if fps > 0 else 0

cap.release()

print(f"해상도       : {width} × {height}")
print(f"FPS          : {fps:.2f}")
print(f"프레임 수    : {frame_count}")
print(f"재생 시간    : {duration:.1f}초")

## 6. 첫 프레임과 가상선 확인

기본 가상선은 화면 높이의 **60% 위치**입니다.

In [ ]:
cap = cv2.VideoCapture(VIDEO_PATH)
ok, first_frame = cap.read()
cap.release()

if not ok:
    raise RuntimeError("영상 첫 프레임을 읽지 못했습니다.")

first_rgb = cv2.cvtColor(first_frame, cv2.COLOR_BGR2RGB)

LINE_RATIO = 0.60
line_y = int(height * LINE_RATIO)

preview = first_rgb.copy()
cv2.line(preview, (0, line_y), (width, line_y), (255, 0, 0), 4)

plt.figure(figsize=(14, 8))
plt.imshow(preview)
plt.axis("off")
plt.title(f"가상선 위치: 화면 높이의 {LINE_RATIO*100:.0f}%")
plt.show()

## 7. 한 프레임에서 사람 탐지해 보기

`classes=[0]`은 사람만 탐지한다는 뜻입니다.

In [ ]:
test_model, _ = load_yolo_model()

results = test_model.predict(
    first_frame,
    classes=[0],
    conf=0.35,
    verbose=False,
    device=DEVICE
)

detected_frame = results[0].plot()
detected_rgb = cv2.cvtColor(detected_frame, cv2.COLOR_BGR2RGB)

plt.figure(figsize=(14, 8))
plt.imshow(detected_rgb)
plt.axis("off")
plt.title("YOLO 사람 탐지 결과")
plt.show()

print("탐지된 사람 수:", len(results[0].boxes))

# 8. ByteTrack 다중 객체 추적

핵심 코드는 다음과 같습니다.

```python
result = model.track(
    frame,
    persist=True,
    tracker="bytetrack.yaml"
)[0]
```

`persist=True`를 사용하면 연속 프레임에서 Track ID를 유지할 수 있습니다.

## 9. 사람 위치와 가상선 통과 판정

Bounding Box의 **아래쪽 중앙점(bottom-center)** 을 사람 위치로 사용합니다.

```text
         ┌─────────────┐
         │    사람     │
         │             │
         └──────●──────┘
                ↑
          bottom-center
```

가상선 주변에는 작은 완충 영역을 두어 좌표 흔들림으로 인한 중복 카운트를 줄입니다.

## 10. 추적 설정값

In [ ]:
CONF = 0.35
IOU = 0.50

LINE_RATIO = 0.60
LINE_MARGIN = 12

TRACKER = "bytetrack.yaml"

RAW_OUTPUT = "/content/cctv_tracking_raw.mp4"
FINAL_OUTPUT = "/content/cctv_people_count.mp4"
CSV_OUTPUT = "/content/crossing_events.csv"

print("모델:", MODEL_NAME)
print("Tracker:", TRACKER)
print("Confidence:", CONF)

## 11. 전체 영상 추적 + 사람 수 세기

- **DOWN**: 가상선 위 → 아래
- **UP**: 가상선 아래 → 위
- **Unique people**: 가상선을 한 번 이상 통과한 고유 Track ID 수

In [ ]:
def side_of_line(y, line_y, margin):
    # -1: 선 위쪽, 0: 완충 영역, +1: 선 아래쪽
    if y < line_y - margin:
        return -1
    elif y > line_y + margin:
        return 1
    return 0


def process_video(
    input_path,
    output_path,
    tracker="bytetrack.yaml",
    line_ratio=0.60,
    line_margin=12,
    conf=0.35,
    iou=0.50
):
    track_model, used_model_name = load_yolo_model()

    cap = cv2.VideoCapture(input_path)

    fps = cap.get(cv2.CAP_PROP_FPS)
    width = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    height = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    frame_count = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))

    if fps <= 0:
        fps = 30.0

    line_y = int(height * line_ratio)

    fourcc = cv2.VideoWriter_fourcc(*"mp4v")
    writer = cv2.VideoWriter(output_path, fourcc, fps, (width, height))

    track_history = defaultdict(list)
    last_stable_side = {}

    counted_down = set()
    counted_up = set()
    unique_crossers = set()
    crossing_events = []

    frame_idx = 0
    start = time.time()

    while cap.isOpened():
        ok, frame = cap.read()
        if not ok:
            break

        result = track_model.track(
            frame,
            persist=True,
            tracker=tracker,
            classes=[0],
            conf=conf,
            iou=iou,
            verbose=False,
            device=DEVICE
        )[0]

        annotated = frame.copy()

        cv2.line(annotated, (0, line_y), (width, line_y), (0, 255, 255), 3)
        cv2.line(annotated, (0, line_y-line_margin), (width, line_y-line_margin), (150,150,150), 1)
        cv2.line(annotated, (0, line_y+line_margin), (width, line_y+line_margin), (150,150,150), 1)

        current_people = 0

        if result.boxes is not None and result.boxes.is_track and result.boxes.id is not None:
            boxes = result.boxes.xyxy.cpu().numpy()
            ids = result.boxes.id.int().cpu().tolist()
            confs = result.boxes.conf.cpu().numpy()

            current_people = len(ids)

            for box, track_id, score in zip(boxes, ids, confs):
                x1, y1, x2, y2 = map(int, box)

                cx = int((x1 + x2) / 2)
                cy = int(y2)

                cv2.rectangle(annotated, (x1, y1), (x2, y2), (0, 255, 0), 2)

                label = f"ID {track_id}  {score:.2f}"
                cv2.putText(
                    annotated, label,
                    (x1, max(25, y1 - 8)),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65,
                    (0, 255, 0), 2
                )

                cv2.circle(annotated, (cx, cy), 5, (0, 0, 255), -1)

                history = track_history[track_id]
                history.append((cx, cy))
                if len(history) > 40:
                    history.pop(0)

                if len(history) >= 2:
                    pts = np.array(history, dtype=np.int32).reshape((-1, 1, 2))
                    cv2.polylines(annotated, [pts], False, (255, 100, 0), 2)

                current_side = side_of_line(cy, line_y, line_margin)

                if current_side != 0:
                    previous_side = last_stable_side.get(track_id)

                    if previous_side is not None and previous_side != current_side:

                        if previous_side == -1 and current_side == 1:
                            if track_id not in counted_down:
                                counted_down.add(track_id)
                                unique_crossers.add(track_id)
                                crossing_events.append({
                                    "track_id": track_id,
                                    "direction": "DOWN",
                                    "frame": frame_idx,
                                    "time_sec": round(frame_idx / fps, 2)
                                })

                        elif previous_side == 1 and current_side == -1:
                            if track_id not in counted_up:
                                counted_up.add(track_id)
                                unique_crossers.add(track_id)
                                crossing_events.append({
                                    "track_id": track_id,
                                    "direction": "UP",
                                    "frame": frame_idx,
                                    "time_sec": round(frame_idx / fps, 2)
                                })

                    last_stable_side[track_id] = current_side

        # 정보 패널
        cv2.rectangle(annotated, (15, 15), (430, 155), (0, 0, 0), -1)

        cv2.putText(
            annotated, f"Current persons : {current_people}",
            (30, 48), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
            (255, 255, 255), 2
        )
        cv2.putText(
            annotated, f"DOWN crossings : {len(counted_down)}",
            (30, 82), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
            (0, 255, 255), 2
        )
        cv2.putText(
            annotated, f"UP crossings   : {len(counted_up)}",
            (30, 116), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
            (0, 255, 255), 2
        )
        cv2.putText(
            annotated, f"Unique people  : {len(unique_crossers)}",
            (30, 148), cv2.FONT_HERSHEY_SIMPLEX, 0.72,
            (0, 255, 0), 2
        )

        writer.write(annotated)
        frame_idx += 1

        if frame_idx % 100 == 0:
            pct = (frame_idx / frame_count * 100) if frame_count else 0
            print(
                f"{frame_idx}/{frame_count} frames "
                f"({pct:.1f}%) | unique={len(unique_crossers)}"
            )

    cap.release()
    writer.release()

    elapsed = time.time() - start

    events_df = pd.DataFrame(
        crossing_events,
        columns=["track_id", "direction", "frame", "time_sec"]
    )

    summary = {
        "model": used_model_name,
        "frames": frame_idx,
        "fps": fps,
        "down": len(counted_down),
        "up": len(counted_up),
        "unique_people": len(unique_crossers),
        "elapsed_sec": elapsed
    }

    return events_df, summary


events_df, summary = process_video(
    VIDEO_PATH,
    RAW_OUTPUT,
    tracker=TRACKER,
    line_ratio=LINE_RATIO,
    line_margin=LINE_MARGIN,
    conf=CONF,
    iou=IOU
)

events_df.to_csv(CSV_OUTPUT, index=False)

print("\n✅ 처리 완료")
print(summary)

## 12. 최종 집계 결과 확인

In [ ]:
print("========== CCTV 사람 통과 결과 ==========")
print("DOWN 방향 :", summary["down"], "회")
print("UP 방향   :", summary["up"], "회")
print("고유 통과 인원 :", summary["unique_people"], "명")
print("처리 시간 :", round(summary["elapsed_sec"], 1), "초")

display(events_df)

## 13. 방향별 통과 횟수 그래프

In [ ]:
count_df = pd.DataFrame({
    "Direction": ["DOWN", "UP"],
    "Count": [summary["down"], summary["up"]]
})

plt.figure(figsize=(7, 4))
plt.bar(count_df["Direction"], count_df["Count"])
plt.title("Line Crossing Count")
plt.xlabel("Direction")
plt.ylabel("Count")
plt.show()

## 14. 브라우저 재생용 H.264 MP4로 변환

In [ ]:
cmd = [
    "ffmpeg", "-y",
    "-i", RAW_OUTPUT,
    "-vcodec", "libx264",
    "-pix_fmt", "yuv420p",
    "-movflags", "+faststart",
    FINAL_OUTPUT
]

result = subprocess.run(
    cmd,
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE
)

if result.returncode == 0 and os.path.exists(FINAL_OUTPUT):
    print("✅ H.264 변환 성공:", FINAL_OUTPUT)
else:
    FINAL_OUTPUT = RAW_OUTPUT
    print("⚠️ 변환 실패: OpenCV 원본 결과 영상을 사용합니다.")

## 15. 결과 영상 재생

In [ ]:
from IPython.display import Video, display

display(Video(FINAL_OUTPUT, embed=True, width=900))

## 16. 결과 파일 다운로드

- `cctv_people_count.mp4` : 추적/카운트 결과 영상
- `crossing_events.csv` : 통과 ID, 방향, 시간 기록

In [ ]:
from google.colab import files

files.download(FINAL_OUTPUT)
files.download(CSV_OUTPUT)

# 🔍 통과 이벤트 CSV 예시

| track_id | direction | frame | time_sec |
|---:|---|---:|---:|
| 3 | DOWN | 152 | 5.07 |
| 7 | UP | 244 | 8.13 |
| 11 | DOWN | 390 | 13.00 |

이 로그를 이용하면 다음 분석으로 확장할 수 있습니다.

- 1분당 방문객 수
- 시간대별 입출입 인원
- 입장/퇴장 비율
- 혼잡 시간대 분석
- 특정 시간 이후 방문객 수

# 🧪 실습 미션 1 — 가상선 위치 바꾸기

현재:

```python
LINE_RATIO = 0.60
```

다음 값을 시험해 보세요.

- `0.40` : 화면 위쪽
- `0.50` : 화면 중앙
- `0.70` : 화면 아래쪽

# 🧪 실습 미션 2 — Tracker 비교

기본:

```python
TRACKER = "bytetrack.yaml"
```

다음처럼 바꿔 볼 수 있습니다.

```python
TRACKER = "botsort.yaml"
```

비교할 항목:
- 사람이 겹칠 때 ID 유지
- 가려졌다가 다시 등장할 때 ID 유지
- 빠르게 움직일 때 ID 안정성

# 🧪 실습 미션 3 — 현재 사람 수와 누적 통과 인원 비교

**현재 사람 수**는 현재 프레임에 보이는 사람 수이고,  
**누적 통과 인원**은 가상선을 실제로 지나간 고유 사람 수입니다.

예를 들어 현재 화면에 4명이 있어도,
누적 통과 인원은 57명일 수 있습니다.

# 🚀 심화 프로젝트 아이디어

1. **상점 방문객 분석** — 입장/퇴장 인원 및 시간대별 그래프
2. **버스·지하철 승하차 분석**
3. **횡단보도 보행량 분석**
4. **공장 안전구역 침입 감지**
5. **안전모 미착용 감지**
6. **ROI 체류시간 분석**
7. **실시간 혼잡도 경고**

### 핵심 구조

`Object Detection → Multi-Object Tracking → Track ID → Trajectory → Line Crossing → People Counting`

# 📌 핵심 코드 정리

### 사람만 추적

```python
result = model.track(
    frame,
    persist=True,
    tracker="bytetrack.yaml",
    classes=[0]
)[0]
```

### Track ID

```python
track_ids = result.boxes.id.int().cpu().tolist()
```

### Bounding Box

```python
boxes = result.boxes.xyxy.cpu().numpy()
```

### 사람의 바닥 중심점

```python
cx = (x1 + x2) / 2
cy = y2
```

### 가상선 통과

```text
위 → 아래 : DOWN + 1
아래 → 위 : UP + 1
```

이번 실습은 실제 CCTV 분석의 핵심인  
**탐지 → 추적 → 궤적 → 이벤트 판정 → 집계** 흐름을 한 번에 경험하도록 구성했습니다.